In [ ]:
# ---------- Token kinds ----------
# We use plain strings as "kinds" so they are easy to read.
NUMBER  = "NUMBER"   # e.g.  42
IDENT   = "IDENT"    # e.g.  x  or  myVar
PLUS    = "PLUS"     # +
MINUS   = "MINUS"    # -
STAR    = "STAR"     # *
SLASH   = "SLASH"    # /
EQUALS  = "EQUALS"   # =
LPAREN  = "LPAREN"   # (
RPAREN  = "RPAREN"   # )
PRINT   = "PRINT"    # the keyword  print
NEWLINE = "NEWLINE"  # end of a statement
EOF     = "EOF"      # end of the whole file

In [ ]:
# ---------- Token ----------
# A Token is just a small bundle of three pieces of information.
class Token:
    def __init__(self, kind, lexeme, line):
        self.kind   = kind    # what TYPE of token is this?
        self.lexeme = lexeme  # the exact text from the source
        self.line   = line    # which line it appeared on (for error messages)

    def __repr__(self):
        # __repr__ controls how the object looks when you print() it
        return f"Token({self.kind}, {repr(self.lexeme)}, line={self.line})"


In [ ]:
# ---------- Lexer ----------
class Lexer:
    def __init__(self, source):
        self.source  = source   # the entire source code as one string
        self.pos     = 0        # index of the character we are looking at right now
        self.line    = 1        # current line number (starts at 1)
        self.tokens  = []       # we will fill this list with Token objects

    # --- helpers ---

    def current_char(self):
        """Return the character at self.pos, or '' if we are past the end."""
        if self.pos < len(self.source):
            return self.source[self.pos]
        return ""

    def advance(self):
        """Move forward one character and return the character we just passed."""
        ch = self.current_char()
        self.pos += 1
        if ch == "\n":
            self.line += 1
        return ch

    def peek(self):
        """Look one character ahead WITHOUT moving forward."""
        next_pos = self.pos + 1
        if next_pos < len(self.source):
            return self.source[next_pos]
        return ""
        # --- main tokenize method ---

    def tokenize(self):
        """
        Walk through every character in self.source and produce tokens.
        Returns a list of Token objects ending with an EOF token.
        """
        while self.pos < len(self.source):
            ch = self.current_char()

            # --- skip spaces and tabs (but NOT newlines) ---
            if ch in (" ", "\t"):
                self.advance()

            # --- skip single-line comments  // ... ---
            elif ch == "/" and self.peek() == "/":
                while self.current_char() not in ("\n", ""):
                    self.advance()

            # --- newline = end of statement ---
            elif ch == "\n":
                self.tokens.append(Token(NEWLINE, "\\n", self.line))
                self.advance()

            # --- single-character tokens ---
            elif ch == "+":
                self.tokens.append(Token(PLUS,   "+", self.line))
                self.advance()
            elif ch == "-":
                self.tokens.append(Token(MINUS,  "-", self.line))
                self.advance()
            elif ch == "*":
                self.tokens.append(Token(STAR,   "*", self.line))
                self.advance()
            elif ch == "/":
                self.tokens.append(Token(SLASH,  "/", self.line))
                self.advance()
            elif ch == "=":
                self.tokens.append(Token(EQUALS, "=", self.line))
                self.advance()
            elif ch == "(":
                self.tokens.append(Token(LPAREN, "(", self.line))
                self.advance()
            elif ch == ")":
                self.tokens.append(Token(RPAREN, ")", self.line))
                self.advance()

            # --- numbers: one or more digits ---
            elif ch.isdigit():
                self.tokens.append(self.read_number())

            # --- identifiers and keywords ---
            elif ch.isalpha() or ch == "_":
                self.tokens.append(self.read_ident_or_keyword())

            # --- anything else is an error ---
            else:
                print(f"[Lexer Error] Unknown character '{ch}' on line {self.line}")
                self.advance()

        # Always end with an EOF token
        self.tokens.append(Token(EOF, "", self.line))
        return self.tokens

    def read_number(self):
        """Collect consecutive digit characters and return a NUMBER token."""
        start_line = self.line
        number_str = ""
        while self.current_char().isdigit():
            number_str += self.advance()
        return Token(NUMBER, number_str, start_line)

    def read_ident_or_keyword(self):
        """
        Collect a word (letters, digits, underscores).
        If the word is a keyword like 'print', return the right token kind.
        """
        start_line = self.line
        word = ""
        while self.current_char().isalnum() or self.current_char() == "_":
            word += self.advance()

        # Check if the word is a keyword
        if word == "print":
            return Token(PRINT, word, start_line)

        # Otherwise it is a variable name
        return Token(IDENT, word, start_line)



In [ ]:
# ============================================================
#  PART 1 — AST Node classes
#
#  Each kind of "thing" in our language gets its own class.
#  All nodes inherit from ASTNode (their shared "parent" class).
# ============================================================

class ASTNode:
    """Base class for every node in the tree. Does nothing on its own."""
    pass


class NumberNode(ASTNode):
    """Represents a literal integer, e.g.  42"""
    def __init__(self, value):
        self.value = value          # store the integer value (an int, not a string)

    def __repr__(self):
        return f"NumberNode({self.value})"


class VarNode(ASTNode):
    """Represents a variable reference, e.g.  x"""
    def __init__(self, name):
        self.name = name            # the variable's name as a string

    def __repr__(self):
        return f"VarNode({self.name})"


class BinOpNode(ASTNode):
    """
    Represents a binary operation, e.g.  x + 5  or  a * (b - 1)
    It has a left child, an operator, and a right child.
    """
    def __init__(self, left, op, right):
        self.left  = left           # an ASTNode (the left side)
        self.op    = op             # the operator as a string: "+", "-", "*", "/"
        self.right = right          # an ASTNode (the right side)

    def __repr__(self):
        return f"BinOpNode({self.left}, '{self.op}', {self.right})"


class AssignNode(ASTNode):
    """
    Represents an assignment statement, e.g.  x = 10 + y
    It stores the variable name (a string) and the expression (an ASTNode).
    """
    def __init__(self, name, expr):
        self.name = name            # the variable being assigned to
        self.expr = expr            # the right-hand-side expression node

    def __repr__(self):
        return f"AssignNode({self.name}, {self.expr})"


class PrintNode(ASTNode):
    """
    Represents a print statement, e.g.  print x + 1
    It stores the expression whose value should be printed.
    """
    def __init__(self, expr):
        self.expr = expr            # the expression to print

    def __repr__(self):
        return f"PrintNode({self.expr})"



In [ ]:
# ============================================================
#  PART 2 — The Parser
#
#  The parser reads tokens one at a time and builds the AST.
#
#  Grammar (what our language looks like):
#
#    program → stmt*
#    stmt    → IDENT EQUALS expr NEWLINE
#            | PRINT expr NEWLINE
#            | NEWLINE                   (blank lines are allowed)
#    expr    → term ((PLUS | MINUS) term)*
#    term    → factor ((STAR | SLASH) factor)*
#    factor  → NUMBER
#            | IDENT
#            | LPAREN expr RPAREN
#
#  Each grammar rule becomes ONE method in this class.
# ============================================================

class Parser:
    def __init__(self, tokens):
        self.tokens = tokens        # the full list of tokens from the lexer
        self.pos    = 0             # index of the token we are looking at now

    # ── helpers ──────────────────────────────────────────────

    def current(self):
        """Return the token we are currently looking at."""
        return self.tokens[self.pos]

    def advance(self):
        """
        Consume the current token and move to the next one.
        Returns the token we just consumed.
        """
        token = self.tokens[self.pos]
        # Only move forward if we haven't reached the end
        if self.pos < len(self.tokens) - 1:
            self.pos += 1
        return token

    def expect(self, kind):
        """
        Consume a token of the expected kind.
        If the current token is NOT that kind, raise an error.
        Use this whenever a specific token is required by the grammar.
        """
        token = self.current()
        if token.kind != kind:
            raise SyntaxError(
                f"Line {token.line}: expected {kind} but got {token.kind} ('{token.lexeme}')"
            )
        return self.advance()

    # ── grammar rules ─────────────────────────────────────────

    def parse_program(self):
        """
        program → stmt*

        Parse zero or more statements until we reach the end of the file.
        Returns a list of AST nodes (one per statement).
        """
        statements = []

        while self.current().kind != EOF:
            # Skip blank lines (a NEWLINE with nothing before it)
            if self.current().kind == NEWLINE:
                self.advance()
                continue

            # Parse one statement and add it to our list
            stmt = self.parse_stmt()
            if stmt is not None:
                statements.append(stmt)

        return statements

    def parse_stmt(self):
        """
        stmt → IDENT EQUALS expr NEWLINE   (assignment)
             | PRINT expr NEWLINE           (print)

        Look at the current token to decide which kind of statement this is.
        """
        token = self.current()

        # ── assignment:  x = <expr> ──
        if token.kind == IDENT:
            name = self.advance().lexeme      # consume the variable name
            self.expect(EQUALS)               # consume the '='
            expr = self.parse_expr()          # parse the right-hand side
            self.expect(NEWLINE)              # consume the newline
            return AssignNode(name, expr)

        # ── print:  print <expr> ──
        elif token.kind == PRINT:
            self.advance()                    # consume 'print'
            expr = self.parse_expr()          # parse what to print
            self.expect(NEWLINE)              # consume the newline
            return PrintNode(expr)

        else:
            raise SyntaxError(
                f"Line {token.line}: unexpected token '{token.lexeme}' ({token.kind})"
            )

    def parse_expr(self):
        """
        expr → term ((PLUS | MINUS) term)*

        An expression is one or more terms separated by + or -.
        We handle + and - here because they have lower precedence than * and /.
        """
        # Parse the first term
        left = self.parse_term()

        # Keep going as long as we see a + or -
        while self.current().kind in (PLUS, MINUS):
            op_token = self.advance()         # consume the + or -
            right    = self.parse_term()      # parse the next term
            # Wrap what we have so far in a BinOpNode, then loop again
            left = BinOpNode(left, op_token.lexeme, right)

        return left

    def parse_term(self):
        """
        term → factor ((STAR | SLASH) factor)*

        A term is one or more factors separated by * or /.
        * and / have HIGHER precedence than + and -, so they bind tighter.
        """
        # Parse the first factor
        left = self.parse_factor()

        # Keep going as long as we see a * or /
        while self.current().kind in (STAR, SLASH):
            op_token = self.advance()         # consume the * or /
            right    = self.parse_factor()    # parse the next factor
            left = BinOpNode(left, op_token.lexeme, right)

        return left

    def parse_factor(self):
        """
        factor → NUMBER
               | IDENT
               | LPAREN expr RPAREN

        A factor is the smallest building block: a number, a variable,
        or a parenthesised sub-expression.
        """
        token = self.current()

        # ── integer literal ──
        if token.kind == NUMBER:
            self.advance()
            return NumberNode(int(token.lexeme))   # convert string to int here

        # ── variable name ──
        elif token.kind == IDENT:
            self.advance()
            return VarNode(token.lexeme)

        # ── parenthesised expression:  ( expr ) ──
        elif token.kind == LPAREN:
            self.advance()                    # consume '('
            expr = self.parse_expr()          # parse whatever is inside
            self.expect(RPAREN)               # consume ')'
            return expr                       # return the inner expression directly

        else:
            raise SyntaxError(
                f"Line {token.line}: unexpected token '{token.lexeme}' in expression"
            )

In [ ]:
# ============================================================
#  PART 3 — Pretty-printer
#
#  Walks the AST and prints it as indented text so we can
#  see the tree structure clearly. Very useful for debugging.
# ============================================================

def pretty_print(node, indent=0):
    """
    Print an AST node and all its children with indentation.
    Each level of the tree is indented two more spaces than its parent.
    """
    prefix = "  " * indent      # e.g. indent=2 gives "    "

    if isinstance(node, NumberNode):
        print(f"{prefix}Number: {node.value}")

    elif isinstance(node, VarNode):
        print(f"{prefix}Var: {node.name}")

    elif isinstance(node, BinOpNode):
        print(f"{prefix}BinOp: '{node.op}'")
        pretty_print(node.left,  indent + 1)
        pretty_print(node.right, indent + 1)

    elif isinstance(node, AssignNode):
        print(f"{prefix}Assign: {node.name} =")
        pretty_print(node.expr, indent + 1)

    elif isinstance(node, PrintNode):
        print(f"{prefix}Print:")
        pretty_print(node.expr, indent + 1)

    else:
        print(f"{prefix}Unknown node: {node}")


In [ ]:
# ============================================================
#  PART 4 — Helper: run the full pipeline (lex + parse)
# ============================================================

def parse_source(source):
    """
    Convenience function: takes source code as a string,
    runs the lexer, then runs the parser, and returns the AST.
    """
    # Step 1: lex
    lexer  = Lexer(source)
    tokens = lexer.tokenize()

    # Step 2: parse
    parser     = Parser(tokens)
    statements = parser.parse_program()

    return statements

In [ ]:
TEST_SOURCE = """\
x = 10
y = x + 5 * (3 - 1)
z = (x + y) * 2
print z
print x + 1
"""

if __name__ == "__main__":
    print("=== Source code ===")
    print(TEST_SOURCE)

    print("=== Abstract Syntax Tree ===")
    statements = parse_source(TEST_SOURCE)

    for i, stmt in enumerate(statements):
        print(f"--- Statement {i + 1} ---")
        pretty_print(stmt)
        print()

=== Source code ===
x = 10
y = x + 5 * (3 - 1)
z = (x + y) * 2
print z
print x + 1

=== Abstract Syntax Tree ===
--- Statement 1 ---
Assign: x =
  Number: 10

--- Statement 2 ---
Assign: y =
  BinOp: '+'
    Var: x
    BinOp: '*'
      Number: 5
      BinOp: '-'
        Number: 3
        Number: 1

--- Statement 3 ---
Assign: z =
  BinOp: '*'
    BinOp: '+'
      Var: x
      Var: y
    Number: 2

--- Statement 4 ---
Print:
  Var: z

--- Statement 5 ---
Print:
  BinOp: '+'
    Var: x
    Number: 1

